In [149]:
import pandas as pd
import numpy as np
import skbio.diversity.alpha as sda
from skbio.diversity import alpha_diversity

kaiju_sum = "../results/summary_kaiju/kaiju_summary_complete.tsv"
kaiju_df = pd.read_csv(kaiju_sum, sep="\t", low_memory=False)

In [150]:
kaiju_df_onlytaxonid = kaiju_df.drop(['phylum', 'class', 'order', 'family', 'genus', 'species'], axis=1)
kaiju_df_onlytaxonid.set_index('taxon_id', inplace=True)

a = kaiju_df_onlytaxonid.loc['unclassified']
b = kaiju_df_onlytaxonid.sum().to_frame(name="sum_counts")
combined = pd.concat([a, b], axis=1)

In [151]:
combined['%unclassified'] = combined.unclassified / combined.sum_counts * 100

In [152]:
# Define the metrics to calculate as a dict of {name: function}
metrics = {
    'ShannonDiversity': sda.shannon,
    'AceDiversity': sda.ace,
    'Chao1Diversity': sda.chao1,
    'SimpsonEveness': sda.simpson,
    'GiniDominance': lambda x: sda.gini_index(x, 'rectangles'),
    'GoodsCoverage': sda.goods_coverage
}

# Select only numeric columns
kaiju_df_no_str = kaiju_df.select_dtypes(include=np.number)

# Compute metrics
alpha_diversities = []
for sample in kaiju_df_no_str.columns:
    values = kaiju_df_no_str[sample]
    results = [sample] + [func(values) for func in metrics.values()]
    alpha_diversities.append(results)

# Build DataFrame
alpha_df = pd.DataFrame(alpha_diversities, columns=['Sample'] + list(metrics.keys()))
alpha_df.set_index('Sample', inplace=True)

# Combine with existing dataframe
combined = pd.concat([combined, alpha_df], axis=1)
combined

,unclassified,sum_counts,%unclassified,ShannonDiversity,AceDiversity,Chao1Diversity,SimpsonEveness,GiniDominance,GoodsCoverage
20_E,2075039,2283857,90.856783,0.795013,11711.580365,11364.515719,0.174247,0.998570,0.998613
PRI_18,1853156,2048761,90.452522,0.832993,11819.552310,11503.512126,0.181601,0.998394,0.998432
PRI_17,2467939,3356777,73.521089,2.457626,22666.475861,22798.725825,0.459170,0.987052,0.998677
PRI_23,4463198,7095930,62.897999,3.278899,29433.284543,29795.309345,0.603757,0.980419,0.999327
9.4_ZA,1270086,1513460,83.919364,1.394697,12061.911947,11705.239024,0.295476,0.997204,0.997890
6.2_ZA,1774578,2392268,74.179732,1.948911,14202.999819,14052.064259,0.447426,0.995322,0.998628
PRI_8,2407033,3086994,77.973362,1.987926,21493.031529,21632.182787,0.391180,0.990108,0.998532
7.1_ZA,1210836,1325720,91.334218,0.775665,7390.046560,6987.512615,0.165677,0.998928,0.998531
9.3_ZA,972910,1104040,88.122713,1.015493,8502.389436,8195.093645,0.223075,0.998494,0.997942
8.3_ZA,1033470,1225623,84.322014,1.388566,12206.713289,11628.253883,0.288751,0.996849,0.997360


In [153]:
combined.to_csv('alpha_diversity.csv')